In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [3]:
loader = HFDatasetLoader(
    dataset_name="SetFit/enron_spam",
    split="train",
    filename="datos/enron_spam_clasicacion.csv",
)

In [5]:
df_clasificacion_crudo = loader.load()
df_clasificacion_crudo.head(2)

Repo card metadata block was not found. Setting CardData to empty.


,message_id,text,label,label_text,subject,message,date
0,33214,any software just for 15 $ - 99 $ understandin...,1,spam,any software just for 15 $ - 99 $,understanding oem software\nlead me not into t...,2005-06-18
1,11929,perspective on ferc regulatory action client c...,0,ham,perspective on ferc regulatory action client c...,"19 th , 2 : 00 pm edt\nperspective on ferc reg...",2001-06-19


## Clasificacion

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, precision_score, recall_score

from clases_po import ClasificadorTexto   # versión con alias de columnas y palabras_clave()

CLASES = ["normal", "spam", "toxico"]
COLORES = {"normal": "#1f6fe5", "spam": "#f2661b", "toxico": "#e5384b"}
UMBRAL_TOXICO = 0.5                                  # ajústalo después de ver el bloque 1
CACHE_TOXICIDAD = Path("datos/enron_toxicidad.csv")  # el modelo de toxicidad corre una sola vez
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

df = df_clasificacion_crudo.copy()
df["text"] = df["text"].fillna("").astype(str)

### 1 · Crear la tercera clase: tóxico  ->  escena 9 (la tercera etiqueta)

In [8]:
%pip install torch

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 1.0/124.1 MB 5.6 MB/s eta 0:00:22
    --------------------------------------- 2.9/124.1 MB 7.3 MB/s eta 0:00:17
   - -------------------------------------- 3.9/124.1 MB 7.8 MB/s eta 0:00:16
   -- ------------------------------------- 6.3/124.1 MB 7.9 MB/s eta 0:00:15
   -- ------------------------------------- 7.6/124.1 MB 7.6 MB/s eta 0:00:16
   -- ------------------------------------- 8.9/124.1 MB 7.3 MB/s eta 0:00:16
   --- ------------------------------------ 11.0/124.1 MB 7.9 MB/s eta 0:00:15
   ---- ----------------------------------- 13.1/124.1 MB 8.1 MB/s eta 0:00:14
   ---- -------------------------------

In [10]:
%pip install transformers

  Using cached transformers-5.20.2-py3-none-any.whl.metadata (31 kB)
  Using cached regex-2026.9.29-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.27.3-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached transformers-5.20.2-py3-none-any.whl (12.8 MB)
Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl (2.9 MB)
Using cached regex-2026.9.29-cp312-cp312-win_amd64.whl (280 kB)
Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)
Using cached typer-0.27.3-py3-none-any.whl (123 kB)
Us

In [ ]:
# Requiere transformers y torch (ya instalados para la voz). Con GPU tarda unos minutos;
# con CPU puede tardar bastante más: por eso el resultado se guarda en CACHE_TOXICIDAD.
if CACHE_TOXICIDAD.exists():
    tox = pd.read_csv(CACHE_TOXICIDAD)
    print("Probabilidades de toxicidad leídas de", CACHE_TOXICIDAD)
else:
    import torch
    from transformers import pipeline
    detector = pipeline("text-classification", model="unitary/toxic-bert", top_k=None,
                        function_to_apply="sigmoid", truncation=True, max_length=512,
                        device=0 if torch.cuda.is_available() else -1)
    ham = df[df["label_text"] == "ham"]
    salida = detector(ham["text"].tolist(), batch_size=32)
    # toxic-bert da seis puntajes (toxic, severe_toxic, obscene, threat, insult, identity_hate):
    # usamos el mayor, para atrapar también amenazas e insultos.
    tox = pd.DataFrame({"message_id": ham["message_id"].to_numpy(),
                        "p_toxico": [max(r["score"] for r in s) for s in salida]})
    CACHE_TOXICIDAD.parent.mkdir(parents=True, exist_ok=True)
    tox.to_csv(CACHE_TOXICIDAD, index=False)
    print("Guardado en", CACHE_TOXICIDAD)

df = df.drop(columns=["p_toxico"], errors="ignore").merge(tox, on="message_id", how="left")
df["p_toxico"] = df["p_toxico"].fillna(0.0)          # el spam no pasa por el detector

for u in [0.3, 0.5, 0.7, 0.9]:
    print(f"umbral {u}: {((df['label_text'] == 'ham') & (df['p_toxico'] >= u)).sum():>5} correos ham serían tóxicos")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(df.loc[df["label_text"] == "ham", "p_toxico"], bins=50, color=COLORES["toxico"])
ax.axvline(UMBRAL_TOXICO, color="#151b6e", ls="--", label=f"umbral {UMBRAL_TOXICO}")
ax.set_yscale("log")
ax.set_title("P(tóxico) en los correos ham (escala logarítmica)")
ax.set_xlabel("P(tóxico | correo)")
ax.legend()
plt.show()

df["etiqueta"] = np.select(
    [df["label_text"] == "spam", df["p_toxico"] >= UMBRAL_TOXICO], ["spam", "toxico"], default="normal")
print(df["etiqueta"].value_counts())

print("\nEjemplos marcados como tóxicos (revísalos: el detector también se equivoca):")
pd.set_option("display.max_colwidth", 160)
print(df[df["etiqueta"] == "toxico"].nlargest(8, "p_toxico")[["p_toxico", "text"]]
      .assign(text=lambda d: d["text"].str[:150]).to_string())

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1263.69it/s]


### 2 · Estructura y calidad  ->  escena 6 (shape, info, faltantes)

In [ ]:
print("Forma (filas, columnas):", df.shape)
df.info()
print("\nValores faltantes por columna:\n", df.isna().sum())
print("\nTextos duplicados exactos:", df.duplicated(subset=["text"]).sum())
print("Correos con texto vacío:", (df["text"].str.strip() == "").sum())
print("Correos sin cuerpo (message vacío):", (df["message"].str.strip() == "").sum())

### 3 · Balance de las tres clases  ->  escena 10 (desbalance y Pere)

In [ ]:
conteo = df["etiqueta"].value_counts().reindex(CLASES).fillna(0).astype(int)
porc = (conteo / conteo.sum() * 100).round(2)
print(pd.DataFrame({"correos": conteo, "%": porc}))

fig, ax = plt.subplots(figsize=(6.5, 3.5))
barras = ax.bar(conteo.index, conteo.values, color=[COLORES[c] for c in conteo.index])
ax.bar_label(barras, labels=[f"{v:,}  ({p} %)" for v, p in zip(conteo.values, porc.values)])
ax.set_title("Correos por clase")
ax.set_ylabel("cantidad")
plt.show()

# Pere, el modelo perezoso: siempre responde la clase más frecuente
mayoritaria = conteo.idxmax()
print(f"Si Pere responde siempre '{mayoritaria}': exactitud = {conteo.max() / conteo.sum():.1%}, "
      f"tóxicos detectados = 0 de {conteo['toxico']}")

### 4 · Longitud de los correos por clase  ->  escenas 5 y 6

In [ ]:
df["palabras"] = df["text"].str.split().str.len()
print(df.groupby("etiqueta")["palabras"].describe().reindex(CLASES).round(1))

tope = df["palabras"].quantile(0.99)
fig, ax = plt.subplots(figsize=(7, 3.5))
for clase in CLASES:
    grupo = df.loc[df["etiqueta"] == clase, "palabras"].clip(upper=tope)
    ax.hist(grupo, bins=50, alpha=.6, density=True, label=clase, color=COLORES[clase])
ax.set_title(f"Palabras por correo (proporción; recortado en el percentil 99 = {tope:.0f})")
ax.set_xlabel("palabras")
ax.legend()
plt.show()

### 5 · Fechas por clase  ->  escena 7 (radiografía): no usar la fecha como característica

In [ ]:
df["anio"] = pd.to_datetime(df["date"], errors="coerce").dt.year
tabla_anios = pd.crosstab(df["anio"], df["etiqueta"]).reindex(columns=CLASES, fill_value=0)
print(tabla_anios)
tabla_anios.plot(kind="bar", stacked=True, figsize=(7, 3.5),
                 color=[COLORES[c] for c in CLASES], title="Correos por año y clase")
plt.ylabel("correos")
plt.show()

### 6 · Palabras más frecuentes por clase  ->  escena 12 (TF-IDF)

In [ ]:
def top_palabras(textos, n=15):
    cv = CountVectorizer(stop_words="english", min_df=3)
    m = cv.fit_transform(textos)
    suma = np.asarray(m.sum(axis=0)).ravel()
    orden = suma.argsort()[::-1][:n]
    return pd.Series(suma[orden], index=cv.get_feature_names_out()[orden])

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for eje, clase in zip(ejes, CLASES):
    top = top_palabras(df.loc[df["etiqueta"] == clase, "text"])
    eje.barh(top.index[::-1], top.values[::-1], color=COLORES[clase])
    eje.set_title(f"Palabras más frecuentes · {clase}")
plt.tight_layout()
plt.show()

### 7 · Primer experimento con la clase (Naive Bayes, tres clases)  ->  escenas 13 y 15

In [ ]:
nb = ClasificadorTexto(modelo=MultinomialNB(), columna_texto="text", columna_objetivo="etiqueta")
r_nb = nb.ejecutar(df)
r_nb["metricas"]["por_clase"].round(4)

### 8 · Palabras que delatan cada clase  ->  escena 14b

In [ ]:
nb.palabras_clave(15)

### 9 · Matriz de confusión 3 × 3  ->  escenas 15 a 21

In [ ]:
ConfusionMatrixDisplay.from_predictions(nb.y_test, r_nb["predicciones"], labels=CLASES,
                                        cmap="Blues", values_format=",d")
plt.title("Matriz de confusión · Naive Bayes")
plt.show()

<!-- ### 10 · Ejemplos de errores  ->  discusión en clase -->

### 10 · Ejemplos de errores  ->  discusión en clase

In [ ]:
errores = pd.DataFrame({"texto": nb.X_test.to_numpy(), "real": nb.y_test.to_numpy(), "predicho": r_nb["predicciones"]})
errores = errores[errores["real"] != errores["predicho"]]
print("Errores por tipo:\n", errores.groupby(["real", "predicho"]).size(), "\n")
for real, pred in [("toxico", "normal"), ("normal", "toxico"), ("normal", "spam"), ("spam", "normal")]:
    sub = errores[(errores["real"] == real) & (errores["predicho"] == pred)]
    print(f"--- real {real} → predicho {pred} ({len(sub)})")
    print(sub["texto"].str[:150].head(3).to_string(), "\n")

### 11 · Cambiar de modelo: regresión logística balanceada  ->  escenas 22 a 24

In [ ]:
lr = ClasificadorTexto(modelo=LogisticRegression(class_weight="balanced", max_iter=1000),
                       columna_texto="text", columna_objetivo="etiqueta")
r_lr = lr.ejecutar(df)

ConfusionMatrixDisplay.from_predictions(lr.y_test, r_lr["predicciones"], labels=CLASES,
                                        cmap="Greens", values_format=",d")
plt.title("Matriz de confusión · Regresión logística")
plt.show()

def resumen(m):
    pc = m["por_clase"]
    return {"exactitud": m["exactitud"], "F1 macro": m["f1_macro"], "F1 ponderado": m["f1_ponderado"],
            "precisión tóxico": pc.loc["toxico", "precision"], "recall tóxico": pc.loc["toxico", "recall"],
            "F1 tóxico": pc.loc["toxico", "f1"], "F1 normal": pc.loc["normal", "f1"], "F1 spam": pc.loc["spam", "f1"]}

comparacion = pd.DataFrame({"Naive Bayes": resumen(r_nb["metricas"]),
                            "Regresión logística": resumen(r_lr["metricas"])}).T.round(4)
print(comparacion)
comparacion.T.plot(kind="bar", figsize=(10, 3.8), ylim=(0, 1.05),
                   color=["#8a3ff0", "#10a179"], title="Naive Bayes vs. regresión logística")
plt.xticks(rotation=20)
plt.show()

### 12 · El umbral de tóxico  ->  escena 25 (el umbral también decide)

In [ ]:
p_tox = lr.predecir_proba(lr.X_test)["toxico"].to_numpy()
es_tox = (lr.y_test == "toxico").to_numpy()
es_normal = (lr.y_test == "normal").to_numpy()

prec, rec, umbrales = precision_recall_curve(es_tox, p_tox)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(umbrales, prec[:-1], label="precisión tóxico", color=COLORES["toxico"])
ax.plot(umbrales, rec[:-1], label="recall tóxico", color="#10a179")
ax.set_xlabel("umbral")
ax.set_title("Precisión y recall de tóxico según el umbral · regresión logística")
ax.legend()
plt.show()

filas = []
for u in [0.2, 0.3, 0.5, 0.7]:
    pred_u = p_tox >= u
    filas.append({"umbral": u, "precisión tóxico": precision_score(es_tox, pred_u, zero_division=0),
                  "recall tóxico": recall_score(es_tox, pred_u),
                  "tóxicos detectados": int((pred_u & es_tox).sum()),
                  "normales acusados": int((pred_u & es_normal).sum())})
print(f"Tóxicos en prueba: {es_tox.sum()}")
print(pd.DataFrame(filas).round(4))

# La clase aplica el umbral directamente:
m_03 = lr.evaluar(lr.X_test, lr.y_test, umbral=0.3, clase_umbral="toxico")

### 13 · ¿Los duplicados inflan el resultado?  ->  escena 7 (radiografía) y discusión

In [ ]:
lr_sin_dup = ClasificadorTexto(modelo=LogisticRegression(class_weight="balanced", max_iter=1000),
                               columna_texto="text", columna_objetivo="etiqueta", eliminar_duplicados=True)
r_sin_dup = lr_sin_dup.ejecutar(df)
print(pd.DataFrame({"con duplicados": resumen(r_lr["metricas"]),
                    "sin duplicados": resumen(r_sin_dup["metricas"])}).T.round(4))